Visual 1: Top 10 Most Sold Products:
- Utilize Product Table (Name) and count PurchaseOrderDetail's OrderQty total 

In [0]:
purchase_order_df = spark.table("hackathon.mini_project_raw.purchasing_purchaseorderdetail")

product_df = spark.table("hackathon.mini_project_raw.production_product")

In [0]:
from pyspark.sql import functions as F

top_10_products_df = (
    purchase_order_df
        .groupBy("productid")
        .agg(F.sum("orderqty").alias("total_qty_sold"))
        .orderBy(F.desc("total_qty_sold"))
        .limit(10)
)

top_10_products_with_details_df = (
    top_10_products_df
        .join(product_df, on="productid", how="left")
        .select("productid", "name", "total_qty_sold")
)

top_10_products_with_details_df.display()


Visual 2. Top 10 Product Subcategories that have the highest average cost to manufacture

In [0]:
product_subcategory_df = spark.table("hackathon.mini_project_raw.production_productsubcategory")

display(product_subcategory_df)

In [0]:
product_category_df = spark.table("hackathon.mini_project_raw.production_productcategory")

display(product_category_df)

In [0]:
top_10_subcategories_df = (
    product_df
        .groupBy("productsubcategoryid")
        .agg(F.avg("standardcost").alias("avg_cost_to_manufacture"))
        .orderBy(F.desc("avg_cost_to_manufacture"))
        .limit(10)
)

top_10_subcategories_with_names_df = (
    top_10_subcategories_df
        .join(product_subcategory_df, on="productsubcategoryid", how="left")
        .select(
            "productsubcategoryid",
            "name",
            "avg_cost_to_manufacture"
        )
)

top_10_subcategories_with_names_df.display()

Visual 3 - Time series chart comparing the monthly sales of product categories.

In [0]:
purchase_order_header_df = spark.table("hackathon.mini_project_raw.purchasing_purchaseorderheader")

display(purchase_order_header_df)

In [0]:
# 1) Build a clean fact table: (order_date, category_name, sales_amount)
sales_by_category_daily_df = (
    purchase_order_header_df.alias("h")
    .join(purchase_order_df.alias("d"), on="purchaseorderid", how="inner")  # assumes both have purchaseorderid
    .join(product_df.alias("p"), on="productid", how="inner")
    .join(product_subcategory_df.alias("sc"), on="productsubcategoryid", how="inner")
    .join(product_category_df.alias("c"), on="productcategoryid", how="inner")
    .withColumn("order_date", F.to_date(F.col("orderdate")))
    .withColumn("sales_amount", F.col("unitprice") * F.col("orderqty"))
    .groupBy("orderdate", F.col("c.name").alias("category_name"))
    .agg(F.sum("sales_amount").alias("daily_sales"))
    .orderBy("orderdate", "category_name")
)

# 2) Pivot to wide format for plotting (one line per category)
sales_pivot_df = (
    sales_by_category_daily_df
    .groupBy("orderdate")
    .pivot("category_name")
    .agg(F.sum("daily_sales"))
    .orderBy("orderdate")
)

# 3) Convert to pandas and plot
pdf = sales_pivot_df.toPandas()

In [0]:
import pandas as pd
import plotly.express as px

# Ensure correct dtypes
pdf["orderdate"] = pd.to_datetime(pdf["orderdate"])
cols = ["Accessories", "Clothing", "Components"]
for c in cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
    
monthly_df = (
    pdf.set_index("orderdate")[cols]
      .fillna(0)
      .resample("MS")   # month start frequency
      .sum()
      .reset_index()
)

# Long format for Plotly
monthly_long = monthly_df.melt(
    id_vars="orderdate",
    value_vars=cols,
    var_name="category",
    value_name="sales"
)

fig = px.line(
    monthly_long,
    x="orderdate",
    y="sales",
    color="category",
    title="Monthly Sales by Product Category",
    line_shape="spline"   # <-- this makes the lines smooth/curved
)
fig.show()


In [0]:
monthly_smoothed = (
    monthly_df.set_index("orderdate")[cols]
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index()
)

monthly_smoothed_long = monthly_smoothed.melt(
    id_vars="orderdate",
    value_vars=cols,
    var_name="category",
    value_name="sales"
)

fig = px.line(
    monthly_smoothed_long,
    x="orderdate",
    y="sales",
    color="category",
    title="Monthly Sales by Product Category (3-Month Rolling Avg)",
    line_shape="spline"
)

fig.show()

In [0]:
df_plot